In [0]:
source_path = '/Volumes/ecommerce_project/bronze/source_files/'

data_files = [
    "order_item_refunds.csv",
    "order_items.csv",
    "orders.csv",
    "products.csv",
    "website_pageviews.csv",
    "website_sessions.csv"
]


for file in data_files:
    df = (
        spark.read\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .csv(source_path + file)
    )

    print(f"\nFILE: {file}")
    print("Columns:", df.columns)
    df.printSchema()
    

In [0]:
from pyspark.sql.functions import current_timestamp, col

source_path = '/Volumes/ecommerce_project/bronze/source_files/'

data_files = {
    "order_item_refunds.csv": "order_item_refunds",
    "order_items.csv": "order_items",
    "orders.csv": "orders",
    "products.csv": "products",
    "website_pageviews.csv": "website_pageviews",
    "website_sessions.csv": "website_sessions"
}

for file_name, table_name in data_files.items():

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(source_path + file_name)
        .withColumn("source_file", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"ecommerce_project.bronze.{table_name}")
    )

        print(f"Created: ecommerce_project.bronze.{table_name}")

In [0]:
%sql
show tables in ecommerce_project.bronze;

In [0]:
%sql
SELECT 'order_item_refunds' AS table_name, COUNT(*) AS row_count
FROM ecommerce_project.bronze.order_item_refunds

UNION ALL

SELECT 'order_items', COUNT(*)
FROM ecommerce_project.bronze.order_items

UNION ALL

SELECT 'orders', COUNT(*)
FROM ecommerce_project.bronze.orders

UNION ALL

SELECT 'products', COUNT(*)
FROM ecommerce_project.bronze.products

UNION ALL

SELECT 'website_pageviews', COUNT(*)
FROM ecommerce_project.bronze.website_pageviews

UNION ALL

SELECT 'website_sessions', COUNT(*)
FROM ecommerce_project.bronze.website_sessions;

In [0]:
%sql
SELECT
    'order_item_refunds' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(order_item_refund_id) AS non_null_keys,
    COUNT(DISTINCT order_item_refund_id) AS distinct_keys
FROM ecommerce_project.bronze.order_item_refunds

UNION ALL

SELECT
    'order_items',
    COUNT(*),
    COUNT(order_item_id),
    COUNT(DISTINCT order_item_id)
FROM ecommerce_project.bronze.order_items

UNION ALL

SELECT
    'orders',
    COUNT(*),
    COUNT(order_id),
    COUNT(DISTINCT order_id)
FROM ecommerce_project.bronze.orders

UNION ALL

SELECT
    'products',
    COUNT(*),
    COUNT(product_id),
    COUNT(DISTINCT product_id)
FROM ecommerce_project.bronze.products

UNION ALL

SELECT
    'website_pageviews',
    COUNT(*),
    COUNT(website_pageview_id),
    COUNT(DISTINCT website_pageview_id)
FROM ecommerce_project.bronze.website_pageviews

UNION ALL

SELECT
    'website_sessions',
    COUNT(*),
    COUNT(website_session_id),
    COUNT(DISTINCT website_session_id)
FROM ecommerce_project.bronze.website_sessions;

In [0]:
%sql
-- verify that foreign-key values have matching records in their parent tables.

select 'order_items -> orders' as relationship, count(*) as unmatched_rows
from ecommerce_project.bronze.order_items oi
left anti join ecommerce_project.bronze.orders o
on oi.order_id = o.order_id

union all

select 'order_items -> products', count(*)
from ecommerce_project.bronze.order_items oi
left anti join ecommerce_project.bronze.products p
on oi.product_id = p.product_id

union all

select 'refunds -> order_items', count(*)
from ecommerce_project.bronze.order_item_refunds r
left anti join ecommerce_project.bronze.order_items oi
on r.order_item_id = oi.order_item_id

union all

select 'orders -> sessions', count(*)
from ecommerce_project.bronze.orders o
left anti join ecommerce_project.bronze.website_sessions s
on o.website_session_id = s.website_session_id

union all

select 'pageviews -> sessions', count(*)
from ecommerce_project.bronze.website_pageviews pv
left anti join ecommerce_project.bronze.website_sessions s
on pv.website_session_id = s.website_session_id;

